In [2]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

OASIS_DIR = "../../data/oasis"
df = pd.read_csv(f"{OASIS_DIR}/oasis_model_ready.csv")

# Stratified split
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")

print("\nTrain label balance:\n",
      train_df["label"].value_counts(normalize=True))

# Create splits directory if it doesn't exist
os.makedirs(f"{OASIS_DIR}/splits", exist_ok=True)

# Save splits
train_df.to_csv(f"{OASIS_DIR}/splits/train.csv", index=False)
val_df.to_csv(f"{OASIS_DIR}/splits/val.csv", index=False)
test_df.to_csv(f"{OASIS_DIR}/splits/test.csv", index=False)

print("\nSplit files saved successfully.")

Train: 164  |  Val: 35  |  Test: 36

Train label balance:
 label
0    0.573171
1    0.426829
Name: proportion, dtype: float64

Split files saved successfully.


In [3]:
# The Dataset class: 3D volume → 2.5D, 3-channel slice stack
import torch
from torch.utils.data import Dataset
import nibabel as nib
import numpy as np

class OASISDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = nib.load(row["scan_path"])
        volume = img.get_fdata().squeeze()  # (256, 256, 128)

        mid = volume.shape[2] // 2
        # 2.5D trick: stack 3 neighbouring slices as 3 channels, like an RGB image —
        # lets a standard ResNet-50 (built for 3-channel input) use local 3D context
        # instead of a single flat slice.
        slice_stack = volume[:, :, mid-1:mid+2]  # shape (256, 256, 3)
        slice_stack = np.transpose(slice_stack, (2, 0, 1))  # -> (3, 256, 256)

        # Normalize to 0-1 per scan (min-max), since raw intensity scales vary by scan
        slice_stack = slice_stack.astype(np.float32)
        slice_stack = (slice_stack - slice_stack.min()) / (slice_stack.max() - slice_stack.min() + 1e-8)

        tensor = torch.from_numpy(slice_stack)
        if self.transform:
            tensor = self.transform(tensor)

        label = torch.tensor(row["label"], dtype=torch.long)
        return tensor, label

# Quick sanity check
train_ds = OASISDataset(f"{OASIS_DIR}/splits/train.csv")
img, label = train_ds[0]
print("Image tensor shape:", img.shape)  # should be torch.Size([3, 256, 256])
print("Label:", label.item())
print("Dataset size:", len(train_ds))

Image tensor shape: torch.Size([3, 256, 256])
Label: 0
Dataset size: 164
